<a href="https://colab.research.google.com/github/thalbl/real-time-translator/blob/main/ml_final_project_t.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-Time Translator with AI

Academic project: **STT (Whisper) -> Translation (M2M100) -> TTS (SpeechT5)**
Adapted for Google Colab with browser recording and interactive interface.

| Stage | Model | Purpose |
|-------|-------|---------|
| STT | OpenAI Whisper | Speech -> Text |
| Translation | Meta M2M100 | Text language A -> Text language B |
| TTS | Microsoft SpeechT5 | Text -> Synthesized speech |

In [ ]:
#C2
!pip install -q torch transformers sentencepiece scipy numpy

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU detected {torch.cuda.get_device_name(0)}")
    print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU! Runtime → Change runtime type → T4 GPU")


No GPU! Runtime → Change runtime type → T4 GPU


In [ ]:
# C3 - Clean up old models from memory
import gc
try: del stt
except: pass
try: del translator
except: pass
try: del tts
except: pass
gc.collect()
torch.cuda.empty_cache()
print(f"Memory cleared! Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

NameError: name 'torch' is not defined

In [ ]:
# ============================================================
# C4 - Imports
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import os, base64, tempfile, subprocess
import numpy as np
import torch
from IPython.display import display, Audio as IPAudio, Javascript, HTML, clear_output
import ipywidgets as widgets
import scipy.io.wavfile as wavfile

from transformers import (
    pipeline,
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan,
    SpeechT5Processor,
)

from google.colab import output as colab_output

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE.upper()}")

In [ ]:
# ============================================================
# Cell 5 - Browser Audio Recording (replaces sounddevice for Colab)
# ============================================================

RECORD_JS = """
async function recordAudio(seconds) {
    const stream = await navigator.mediaDevices.getUserMedia({audio: true});
    const recorder = new MediaRecorder(stream, {mimeType: 'audio/webm;codecs=opus'});
    const chunks = [];

    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.start();

    // Visual recording indicator
    const el = document.createElement('div');
    el.style.cssText = `
        padding: 12px 24px; margin: 8px 0;
        background: linear-gradient(135deg, #e74c3c, #c0392b);
        color: #fff; border-radius: 10px; font-size: 15px;
        text-align: center; font-weight: 600;
        box-shadow: 0 4px 15px rgba(231,76,60,0.4);
        animation: recPulse 1.5s ease-in-out infinite;
    `;
    el.textContent = 'Recording... speak now! (' + seconds + 's)';
    const style = document.createElement('style');
    style.textContent = '@keyframes recPulse{0%,100%{transform:scale(1);opacity:1}50%{transform:scale(1.02);opacity:0.7}}';
    document.head.appendChild(style);
    document.querySelector('#output-area').appendChild(el);

    await new Promise(r => setTimeout(r, seconds * 1000));

    const stopped = new Promise(r => recorder.onstop = r);
    recorder.stop();
    await stopped;
    stream.getTracks().forEach(t => t.stop());
    el.remove(); style.remove();

    const blob = new Blob(chunks, {type: 'audio/webm'});
    const reader = new FileReader();
    return await new Promise(r => {
        reader.onloadend = () => r(reader.result.split(',')[1]);
        reader.readAsDataURL(blob);
    });
}
"""


def record_browser_audio(duration_seconds=5):
    """
    Record audio from the browser microphone.
    Returns: np.ndarray (mono, 16kHz, float32)
    """
    print(f"Preparing {duration_seconds}s recording...")
    display(Javascript(RECORD_JS))

    # Capture audio via JavaScript -> base64 (WebM)
    b64_audio = colab_output.eval_js(f'recordAudio({duration_seconds})')
    audio_bytes = base64.b64decode(b64_audio)

    # Save as WebM and convert to WAV 16kHz mono with ffmpeg
    tmp_webm = tempfile.mktemp(suffix='.webm')
    tmp_wav  = tempfile.mktemp(suffix='.wav')

    with open(tmp_webm, 'wb') as f:
        f.write(audio_bytes)

    subprocess.run(
        ['ffmpeg', '-y', '-i', tmp_webm, '-ar', '16000', '-ac', '1', '-f', 'wav', tmp_wav],
        capture_output=True
    )

    sr, audio = wavfile.read(tmp_wav)
    audio = audio.astype(np.float32) / 32768.0  # int16 -> float32

    os.remove(tmp_webm)
    os.remove(tmp_wav)

    print(f"Audio captured: {len(audio)} samples ({len(audio)/16000:.1f}s)")
    return audio


print("record_browser_audio() function ready!")

In [ ]:
# ============================================================
# Cell 6 - STT Module (Speech-to-Text) -- Whisper
# ============================================================
from transformers import WhisperProcessor, WhisperForConditionalGeneration

class STTModule:
    """Converts speech (audio) to text using Whisper."""

    def __init__(self, model_name="openai/whisper-tiny", device=DEVICE):
        print(f"[STT] Loading model '{model_name}' on '{device}'...")
        self.processor = WhisperProcessor.from_pretrained(model_name)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)
        self.device = device
        self.sample_rate = 16000
        print("[STT] Model loaded!")

    def transcribe(self, audio):
        """Transcribe a numpy audio array to text."""
        input_features = self.processor(
            audio,
            sampling_rate=self.sample_rate,
            return_tensors="pt",
        ).input_features.to(self.device)

        predicted_ids = self.model.generate(
            input_features,
            language="portuguese",
        )

        text = self.processor.batch_decode(
            predicted_ids, skip_special_tokens=True
        )[0].strip()
        print(f"[STT] Recognized text: '{text}'")
        return text

    def record_and_transcribe(self, duration_seconds=5.0):
        """Record from the browser microphone and transcribe."""
        audio = record_browser_audio(duration_seconds)
        return self.transcribe(audio)


print("STTModule class defined!")

In [ ]:
# ============================================================
# C9 - Load all AI models
# ============================================================
print("Loading models... this may take a few minutes.\n")

# Re-initializing models to ensure they use the latest class definitions.
stt = STTModule(model_name="openai/whisper-medium", device=DEVICE)
print()

translator = TranslatorModule(source_lang="pt", target_lang="en", model_name="facebook/m2m100_1.2B", device=DEVICE)

print()

tts = TTSModule(device=DEVICE)

print("\n" + "=" * 50)
print("ALL MODELS LOADED SUCCESSFULLY!")
print("=" * 50)

In [ ]:
# ============================================================
# C11 - STT Test
# ============================================================

# Test: Speech-to-Text
print("Recording 5 seconds...")
audio = record_browser_audio(5)
text = stt.transcribe(audio)
print(f"\nResult: {text}")

# Play back the captured audio:
display(IPAudio(audio, rate=16000))